# Conexión Mongo DB

In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['datos_scraping'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


# Conectar el Scraper a la BD y guardar
Se usa el script de la semana 3

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time #para capturar la hora y fecha de obtención

NOMBRE_GRUPO = "Nombre del grupo" #Agegaremos el nombre de grupo dentro del diccionario y la hora en que se obtubo

# --- PASO 1: CONFIGURACIÓN ---
options = Options()
options.add_argument("--headless") 
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
datos_finales = []
limite_paginas = 3

try:
    driver.get("http://books.toscrape.com/")
    
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Nivel de Profundidad {nivel_pagina + 1} ---")
        
        # 1. Espera al Contenedor Raíz (el que agrupa todos los resultados)
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        # 2. Captura de Bloques de Primer Nivel (la "tarjeta" o "card" del item)
        bloques_primer_nivel = driver.find_elements(By.CLASS_NAME, "product_pod")
        
        for bloque in bloques_primer_nivel:
            # 3. Búsqueda Detallada dentro del bloque (Selectores específicos)
            # Extraemos el atributo 'title' del enlace para evitar texto truncado
            dato_identificador = bloque.find_element(By.TAG_NAME, "h3").find_element(By.TAG_NAME, "a").get_attribute("title")
            dato_valor = bloque.find_element(By.CLASS_NAME, "price_color").text
            
            # Guardamos la estructura en nuestra memoria
            item_extraido = {
                "identificador": dato_identificador,
                "valor": dato_valor,
                "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                "grupo": NOMBRE_GRUPO
            }
            datos_finales.append(item_extraido)

        # --- NAVEGACIÓN: Lógica de Salto de Nodo (Paginación) ---
        try:
            # Buscamos el activador del siguiente nodo de datos
            disparador_siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a")
            disparador_siguiente.click()
        except Exception:
            print("Fin del árbol de navegación o nodo no encontrado.")
            break 

    print(f"\nExtracción finalizada. Registros en memoria: {len(datos_finales)}")

except Exception as e:
    print(f"Fallo en la arquitectura de scraping: {e}")

finally:
    driver.quit()

#TENEMOS EL DICCIONARIO CON UN FORMATO JSON, OBTENIDO DEL SCRIPT. AHORA LO INCLUIMOS EN LA BASE DE DATOS
#Cambios en los datos de salida
from pymongo import MongoClient



# 1. Configuracion de conexion
try:
    # 'database' es el nombre del servicio en el docker-compose
    client = MongoClient('database', 27017)
    db = client['proyecto_bigdata']
    coleccion = db['datos_scraping']
    print("CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.")
except Exception as e:
    print("ERROR DE CONEXION:", e)

# 2. Procesamiento e insercion de datos
registros_exitosos = 0
registros_fallidos = 0

for dato in datos_finales:
    try:
        # Limpieza de simbolos y conversion a float
        # Se eliminan simbolos de moneda, comas y espacios en blanco
        valor_sucio = str(dato.get('valor', '0'))
        valor_limpio = valor_sucio.replace('£', '').replace(',', '').strip()
        
        dato['valor'] = float(valor_limpio)
        
        # Insercion individual
        coleccion.insert_one(dato)
        registros_exitosos += 1
        
    except ValueError:
        print("ADVERTENCIA: No se pudo procesar el valor:", valor_sucio)
        registros_fallidos += 1
    except Exception as e:
        print("ERROR EN INSERCION:", e)
        registros_fallidos += 1

print("RESUMEN DE CARGA:")
print("Registros exitosos:", registros_exitosos)
print("Registros fallidos:", registros_fallidos)

--- Procesando Nivel de Profundidad 1 ---
--- Procesando Nivel de Profundidad 2 ---
--- Procesando Nivel de Profundidad 3 ---

Extracción finalizada. Registros en memoria: 60
CONEXION ESTABLECIDA: Conectado a MongoDB exitosamente.
RESUMEN DE CARGA:
Registros exitosos: 60
Registros fallidos: 0


In [1]:
from pymongo import MongoClient

client = MongoClient('mongodb://mongodb:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_scraping']

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get("https://www.google.com")
    print(f"CONECTADO. Título: {driver.title}")
    driver.quit()

except Exception as e:
    print(f"Error: {e}")

CONECTADO. Título: Google


In [3]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

def iniciar_navegador():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

driver = iniciar_navegador()
datos = []

try:
    url = "https://finance.yahoo.com/markets/stocks/gainers/"
    print("Conectando a Yahoo Finance...")
    driver.get(url)
    time.sleep(6)

    filas = driver.find_elements(By.TAG_NAME, "tr")
    print(f"Se encontraron {len(filas)} filas potenciales")

    for fila in filas:
        try:
            columnas = fila.find_elements(By.TAG_NAME, "td")

            if len(columnas) >= 6:
                simbolo = columnas[0].text.strip()
                nombre = columnas[1].text.strip()
                precio = columnas[3].text.strip()
                cambio = columnas[4].text.strip()
                porcentaje = columnas[5].text.strip()

                datos.append({
                    "simbolo": simbolo,
                    "nombre": nombre,
                    "precio": precio,
                    "cambio": cambio,
                    "porcentaje": porcentaje
                })
        except:
            continue

    if datos:
        df = pd.DataFrame(datos)
        df.to_csv("datos_yahoo_finance.csv", index=False)
        print("Proceso exitoso. Archivo generado: datos_yahoo_finance.csv")
        print(df.head())
    else:
        print("No se capturaron datos.")

finally:
    driver.quit()
    print("Navegador cerrado.")

Conectando a Yahoo Finance...
Se encontraron 27 filas potenciales
Proceso exitoso. Archivo generado: datos_yahoo_finance.csv
  simbolo                                     nombre  precio  cambio  \
0    SLNO                  Soleno Therapeutics, Inc.   52.25  +12.76   
1     CAR                    Avis Budget Group, Inc.  212.60  +22.18   
2    BOOT                   Boot Barn Holdings, Inc.  149.03  +13.87   
3    KTOS  Kratos Defense & Security Solutions, Inc.   74.09   +6.78   
4    AMPX                 Amprius Technologies, Inc.   17.56   +1.56   

  porcentaje  
0    +32.31%  
1    +11.65%  
2    +10.26%  
3    +10.07%  
4     +9.75%  
Navegador cerrado.


In [7]:
from pymongo import MongoClient
import time

client = MongoClient('mongodb://bigdata_mongodb:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_scraping']

coleccion.insert_one({
    "texto_crudo": "prueba yahoo",
    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
    "grupo": "DataTraders"
})

print("dato insertado")

dato insertado


In [8]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from pymongo import MongoClient

# ---------------------------
# CONEXIÓN A MONGO
# ---------------------------
client = MongoClient('mongodb://bigdata_mongodb:27017/')
db = client['proyecto_bigdata']
coleccion = db['datos_scraping']

# ---------------------------
# SELENIUM
# ---------------------------
options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

try:
    print("Conectando a Yahoo Finance...")
    driver.get("https://finance.yahoo.com/markets/stocks/gainers/")
    time.sleep(6)

    filas = driver.find_elements(By.TAG_NAME, "tr")
    print(f"Se encontraron {len(filas)} filas")

    for fila in filas:
        try:
            columnas = fila.find_elements(By.TAG_NAME, "td")

            if len(columnas) >= 6:
                simbolo = columnas[0].text.strip()
                nombre = columnas[1].text.strip()
                precio = columnas[3].text.strip()
                cambio = columnas[4].text.strip()
                porcentaje = columnas[5].text.strip()

                documento = {
                    "simbolo": simbolo,
                    "nombre": nombre,
                    "precio": precio,
                    "cambio": cambio,
                    "porcentaje": porcentaje,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": "DataTraders"
                }

                #  GUARDAR EN MONGO
                coleccion.insert_one(documento)

        except:
            continue

    print("Datos guardados en MongoDB")

finally:
    driver.quit()

Conectando a Yahoo Finance...
Se encontraron 27 filas
Datos guardados en MongoDB
